# Download metadata and run all DIST-S1 production checks

Walks through the full validation workflow over an AOI: download DIST-S1 and RTC-S1 metadata
from CMR into geoparquet tables, fetch product tags from the DAAC, run checks 2-6, and write
CSVs plus an interactive HTML report.

The CLI equivalent of this notebook is:

```bash
dist-s1-prod-check run-all --start 2026-01-01 --stop 2026-03-01 --bbox -120 34 -118 36
```

In [ ]:
from pathlib import Path

import pandas as pd

from dist_s1_prod_check import (
    build_rtc_input_frame,
    build_section,
    check_confirmation,
    check_coverage,
    check_duplicates,
    check_inputs_offline,
    check_processing_order,
    deduplicate_products,
    expected_pass_groups,
    fetch_tags_table,
    get_dist_s1_table,
    get_rtc_s1_table_with_lookback,
    resolve_target_tiles,
    write_html_report,
)

In [ ]:
START = '2026-01-01'
STOP = '2026-03-01'
BBOX = (-120.0, 34.0, -118.0, 36.0)
N_TAG_SAMPLE = 40
DATA_DIR = Path('data')
OUT_DIR = Path('outputs')
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

## 1. Download metadata tables from CMR

The RTC-S1 query bbox is widened to cover the full footprint of every burst feeding the
AOI's MGRS tiles, otherwise products at the AOI edge appear to use unexpected inputs.

In [ ]:
target_tiles, rtc_bbox = resolve_target_tiles(BBOX)
print(f'{len(target_tiles)} MGRS tiles, RTC bbox {rtc_bbox}')

df_dist = get_dist_s1_table(START, STOP, bbox=BBOX)
df_dist.to_parquet(DATA_DIR / 'dist_s1_products.parquet', compression='zstd')

df_rtc = get_rtc_s1_table_with_lookback(START, STOP, bbox=rtc_bbox)
df_rtc.to_parquet(DATA_DIR / 'rtc_s1_products.parquet', compression='zstd')

len(df_dist), len(df_rtc)

## 2. Duplicates and processing order (checks 2-3)

In [ ]:
df_dup = check_duplicates(df_dist)
df_dedup = deduplicate_products(df_dist)
df_order = check_processing_order(df_dedup)
n_excluded = int(df_order.is_reprocessed.sum())
print(f'{len(df_dup)} duplicate granules ({n_excluded} acquisitions reprocessed, excluded from ordering)')
print(f'{df_order.out_of_order.sum()} out-of-order products of {int(df_order.evaluated.sum())} evaluated')
df_order[df_order.out_of_order]

## 3. Fetch product tags from the DAAC

Reads `pre_rtc_opera_ids`, `post_rtc_opera_ids`, and `prior_dist_s1_product` from each
product's GEN-DIST-STATUS GeoTIFF. Resumable: already-fetched products are skipped on rerun.

In [ ]:
df_sample = df_dedup.sample(n=min(N_TAG_SAMPLE, len(df_dedup)), random_state=42) if N_TAG_SAMPLE else df_dedup
df_tags = fetch_tags_table(df_sample, DATA_DIR / 'dist_s1_tags.parquet', max_workers=16)
df_tags = df_tags[df_tags.opera_id.isin(df_sample.opera_id)]
df_tags.head()

## 4. Confirmation chain (check 4)

In [ ]:
df_conf = check_confirmation(df_dedup, df_tags)
df_conf = df_conf[df_conf.opera_id.isin(df_tags.opera_id)]
print(f'{(~df_conf.confirmed_ok).sum()} failures of {len(df_conf)}')
df_conf[~df_conf.confirmed_ok]

## 5. Product inputs (check 5, offline)

Re-enumerates every expected product from the local RTC-S1 table with `dist-s1-enumerator`
and compares against the inputs recorded in each product's tags.

In [ ]:
df_frame = build_rtc_input_frame(df_rtc, mgrs_tile_ids=target_tiles)
df_inputs = check_inputs_offline(df_tags, df_frame)
df_inputs[df_inputs.inputs_correct != True]  # noqa: E712

## 6. Coverage: every triggerable product exists (check 6)

In [ ]:
df_groups = expected_pass_groups(df_frame, START, STOP)
df_cov = check_coverage(df_groups, df_dedup, df_rtc_frame=df_frame)
df_missing = df_cov[~df_cov.product_found & df_cov.missing_product_expected]
print(f'{len(df_missing)} missing products of {len(df_cov)} expected passes')
df_missing

## 7. Write the HTML report

In [ ]:
tile_geoms = df_dist[['mgrs_tile_id', 'geometry']].drop_duplicates(subset='mgrs_tile_id')
sections = [
    build_section('duplicates', 'Duplicates', 'Granules sharing tile and acquisition.', df_dup, len(df_dist)),
    build_section('ordering', 'Processing order', 'Out-of-order products.', df_order[df_order.out_of_order], len(df_order)),
    build_section('confirmation', 'Confirmation chain', 'Wrong prior product.', df_conf[~df_conf.confirmed_ok], len(df_conf)),
    build_section('inputs', 'Product inputs', 'Recorded inputs differ from expected.', df_inputs[df_inputs.inputs_correct != True], df_inputs.opera_id.nunique()),  # noqa: E712
    build_section('coverage', 'Coverage', 'Expected products with no granule.', df_missing.drop(columns=['burst_ids', 'burst_pols'], errors='ignore'), len(df_cov)),
]
stamp = pd.Timestamp.now().strftime('%Y%m%dT%H%M%S')
write_html_report(sections, tile_geoms, OUT_DIR / f'report_{stamp}.html', 'DIST-S1 Production Check', f'{START} to {STOP}')